# Queens 8 × 8 · Confidence-guided DFM sampling

Load the trained checkpoint from `01_queens_discrete_flow_matching.ipynb` to study
confidence-weighted erasure. The experiment uses the saved test split and compares
**three temperatures × seven generation-step budgets**.

Run the cells in order to load the model, inspect solution trajectories and plot
the probability of finding at least one solution within a given attempt budget.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from _shared.queens_data import BOARD_SIZE, EMPTY, load_queens_dataset, solution_checks
from _shared.queens_confidence_sampling import (
    load_trained_dfm,
    sample_confidence_dfm,
    retry_statistics,
)
from _shared.queens_sampling_plots import confidence_retry_plot

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens8"
CHECKPOINT_PATH = ARTIFACTS / "queens8_dfm.pt"
RESULT_PATH = ARTIFACTS / "queens8_dfm_confidence_evaluation.npz"
PLOT_CONFIG = dict(responsive=True, displaylogo=False, displayModeBar=False)
NUM_CLASSES = BOARD_SIZE + 1
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#C7B8A3",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / NUM_CLASSES, (index + 1) / NUM_CLASSES)
]

## 1. Load the trained model

Load the prepared dataset, model weights and test indices. The loader verifies
that the checkpoint matches the dataset. The saved architecture determines whether the model uses scalar time.

The model outputs one erasure logit per cell; sigmoid gives the probability
of EMPTY in the endpoint.


In [ ]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens8")
regions = puzzles["regions"]
model, splits, model_info = load_trained_dfm(CHECKPOINT_PATH, puzzles, device=DEVICE)
SEED = model_info["seed"]
TAUS = (0.25, 1.0, 4.0)
EVAL_STEP_COUNTS = (1, 2, 4, 8, 16, 32, 64)
display(
    HTML(
        f"<b>Loaded trained DFM</b> · {DEVICE} · {model_info['optimizer_updates']:,} updates<br>"
        f"Time conditioning: {model_info['use_time_conditioning']}<br>"
        f"{len(regions):,} puzzles · {len(splits['test']):,} held-out puzzles · "
        f"{sum(p.numel() for p in model.parameters()):,} parameters"
    )
)

In [3]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=NUM_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig

## 2. Confidence-weighted erasure

At each step, compute $p_i=\sigma(\ell_i)=P_\theta(x_1^i=\mathrm{EMPTY}\mid x_t,t)$ for every
still-colored cell. Select cells to erase **without replacement**, with weights

$$
w_i=p_i^{1/\tau}.
$$

- **τ = 0.25:** favor high-confidence erasures strongly.
- **τ = 1:** sample proportionally to the predicted empty probability.
- **τ = 4:** flatten the weights and explore more.

Gumbel top-k implements the weighted draw using scores
$\log(p_i)/\tau-\log(-\log u_i)$, where $u_i\sim U(0,1)$.
Already-empty cells are excluded.

An 8×8 board must lose 56 cells. After step $k$ of $S$, the cumulative quota is

$$
d_k=\left\lfloor 56\frac{k}{S}+\frac12\right\rfloor,
\qquad m_k=d_k-d_{k-1}.
$$

Erase $m_k$ cells using the current prediction, then recompute the prediction at
the next step. Exactly eight colored cells remain at the end. The quota fixes
the queen count; evaluation checks rows, columns, regions and non-touching rules.
Quota-constrained erasure defines a heuristic transition kernel.

Every budget makes exactly $S$ network calls at $t=(k-1)/S$. For $S=64$, eight
rounds have a zero deletion quota; their network calls still count toward the budget.


## 3. Solution trajectories

Show two held-out puzzles at each temperature, with one sampled attempt per
temperature. Rows correspond to temperatures; columns show time. Each row starts
from the full coloring, and the final panel marks the remaining cells as queens.


In [4]:
N_TRAJECTORY_PUZZLES = 2
TRAJECTORY_STEPS = 16
CAPTURE_STEPS = tuple(
    sorted(set(np.rint(np.linspace(0, TRAJECTORY_STEPS, 6)).astype(int)))
)
for number, index in enumerate(splits["test"][:N_TRAJECTORY_PUZZLES]):
    board = regions[int(index)]
    trajectory_runs = []
    for tau in TAUS:
        prediction, snapshots = sample_confidence_dfm(
            model,
            board,
            steps=TRAJECTORY_STEPS,
            tau=tau,
            seed=SEED + 600 + number,
            capture_steps=CAPTURE_STEPS,
        )
        solved = bool(solution_checks(board, prediction[0].numpy())["solved"])
        trajectory_runs.append((tau, snapshots, solved))
    titles = [
        (
            f"τ = {tau:g} · t = 0"
            if step == 0
            else (
                ("Solved" if solved else "Not solved")
                if step == TRAJECTORY_STEPS
                else f"t = {step / TRAJECTORY_STEPS:.2f}"
            )
        )
        for tau, _, solved in trajectory_runs
        for step in CAPTURE_STEPS
    ]
    fig = make_subplots(
        rows=len(TAUS),
        cols=len(CAPTURE_STEPS),
        subplot_titles=titles,
        horizontal_spacing=0.025,
        vertical_spacing=0.10,
    )
    for row, (tau, snapshots, solved) in enumerate(trajectory_runs, 1):
        for col, step in enumerate(CAPTURE_STEPS, 1):
            add_board(
                fig,
                snapshots[step][0].numpy(),
                board,
                row,
                col,
                queens=step == TRAJECTORY_STEPS,
            )
    style_boards(
        fig,
        f"Confidence-guided DFM · {puzzles['ids'][int(index)]} · {TRAJECTORY_STEPS} steps",
        rows=len(TAUS),
    ).show(config=PLOT_CONFIG)

## 4. Probability of at least one solution

For each of the **21 (τ, steps) combinations**, run 32 stochastic attempts on
the same 256 held-out puzzles. Shared seeds pair randomness across combinations.
Each attempt starts from the full coloring.

For a puzzle with $c$ successes among $A$ attempts, estimate the chance of at least
one solution within $k$ attempts by averaging over all recorded subsets:

$$
\widehat{\mathrm{pass@}k}=1-\frac{\binom{A-c}{k}}{\binom{A}{k}},
\qquad 0\leq k\leq A.
$$

Average across puzzles. The 95% intervals resample whole puzzles and use the same
bootstrap draws for all combinations. Evaluation saves the individual outcomes
and the dataset/checkpoint hashes.


In [ ]:
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
if (
    not TAUS
    or len(set(TAUS)) != len(TAUS)
    or any(not np.isfinite(t) or t <= 0 for t in TAUS)
):
    raise ValueError("Temperatures must be distinct finite positive values.")
if (
    not EVAL_STEP_COUNTS
    or len(set(EVAL_STEP_COUNTS)) != len(EVAL_STEP_COUNTS)
    or any(not isinstance(s, int) or s < 1 for s in EVAL_STEP_COUNTS)
):
    raise ValueError("Step budgets must be distinct positive integers.")
evaluation_indices = splits["test"].copy()
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)
successes = np.empty(
    (len(TAUS), len(EVAL_STEP_COUNTS), len(evaluation_indices), ATTEMPTS_PER_PUZZLE),
    dtype=bool,
)
evaluation_complete = False
evaluation_progress = widgets.IntProgress(
    value=0,
    max=len(flat_indices) * len(TAUS) * sum(EVAL_STEP_COUNTS),
    description="Evaluation",
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
completed_work = 0
for tau_index, tau in enumerate(TAUS):
    for step_index, generation_steps in enumerate(EVAL_STEP_COUNTS):
        success_batches = []
        for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
            indices = flat_indices[start : start + EVAL_BATCH_SIZE]
            evaluation_status.value = (
                f"τ = {tau:g} · {generation_steps} steps · "
                f"combination {tau_index * len(EVAL_STEP_COUNTS) + step_index + 1}/"
                f"{len(TAUS) * len(EVAL_STEP_COUNTS)} · "
                f"{start:,}/{len(flat_indices):,} trajectories"
            )

            def update_evaluation(
                done,
                total,
                start=start,
                count=len(indices),
                offset=completed_work,
                steps=generation_steps,
            ):
                if done % 5 == 0 or done == total:
                    evaluation_progress.value = offset + start * steps + done * count

            prediction = sample_confidence_dfm(
                model,
                regions[indices],
                steps=generation_steps,
                tau=tau,
                seed=EVAL_SEED + start,
                progress=update_evaluation,
            )
            success_batches.append(
                solution_checks(regions[indices], prediction.numpy())["solved"]
            )
        successes[tau_index, step_index] = np.concatenate(success_batches).reshape(
            len(evaluation_indices),
            ATTEMPTS_PER_PUZZLE,
        )
        completed_work += len(flat_indices) * generation_steps
        evaluation_progress.value = completed_work
evaluation_complete = True
evaluation_progress.bar_style = "success"
evaluation_status.value = (
    f"Completed {len(TAUS) * len(EVAL_STEP_COUNTS)} combinations · "
    f"{len(evaluation_indices):,} puzzles · {ATTEMPTS_PER_PUZZLE} attempts each · "
    f"{len(flat_indices) * len(TAUS) * len(EVAL_STEP_COUNTS):,} trajectories"
)
evaluation_metadata = dict(
    taus=np.asarray(TAUS),
    steps=np.asarray(EVAL_STEP_COUNTS),
    dataset_indices=evaluation_indices,
    puzzle_ids=puzzles["ids"][evaluation_indices],
    evaluation_seed=EVAL_SEED,
    batch_size=EVAL_BATCH_SIZE,
    checkpoint_sha256=model_info["checkpoint_sha256"],
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    optimizer_updates=model_info["optimizer_updates"],
    use_time_conditioning=model_info["use_time_conditioning"],
    sampler_version="queens8_confidence_quota_v1",
    total_erasures=BOARD_SIZE**2 - BOARD_SIZE,
)
np.savez_compressed(RESULT_PATH, successes=successes, **evaluation_metadata)

### Grouped comparison

**Color = τ. Line style = generation steps.** Select temperatures and step budgets
independently to display their combinations. Each selector group has All/None
actions. Enable the 95% bands in Layers; they start hidden to keep the curves
readable. Selection updates the displayed curves immediately.


In [6]:
if not evaluation_complete:
    raise RuntimeError("Finish the evaluation cell before plotting.")
statistics = retry_statistics(successes, bootstrap_samples=3000, seed=EVAL_SEED + 1)
np.savez_compressed(
    RESULT_PATH,
    successes=successes,
    **evaluation_metadata,
    attempt_counts=statistics["attempts"],
    pass_at_k=statistics["mean"],
    lower=statistics["lower"],
    upper=statistics["upper"],
)
confidence_view = confidence_retry_plot(
    statistics,
    TAUS,
    EVAL_STEP_COUNTS,
    previous=globals().get("confidence_view"),
)
display(confidence_view)

SamplingComparison(children=(FigureWidget({
    'data': [{'fillcolor': 'rgba(117,167,255,0.07)',
             …